# Teil 4 - Evaluation

Notebook: `evaluation.ipynb`  
Datensatz: `Steel_industry_data.csv`
        


In [1]:
import csv
from datetime import datetime

from sklearn.feature_extraction import DictVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix, mean_absolute_error, r2_score


def load_rows(path):
    rows = []
    with open(path, newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file, delimiter=";")
        for row in reader:
            timestamp = datetime.strptime(row["date"], "%d.%m.%Y %H:%M")
            rows.append(
                {
                    "Usage_kWh": float(row["Usage_kWh"]),
                    "Lagging_Current_Reactive.Power_kVarh": float(row["Lagging_Current_Reactive.Power_kVarh"]),
                    "Leading_Current_Reactive_Power_kVarh": float(row["Leading_Current_Reactive_Power_kVarh"]),
                    "Lagging_Current_Power_Factor": float(row["Lagging_Current_Power_Factor"]),
                    "Leading_Current_Power_Factor": float(row["Leading_Current_Power_Factor"]),
                    "NSM": int(row["NSM"]),
                    "WeekStatus": row["WeekStatus"],
                    "Day_of_week": row["Day_of_week"],
                    "Month": timestamp.month,
                    "Day": timestamp.day,
                    "Hour": timestamp.hour,
                }
            )
    return rows


rows = load_rows("Steel_industry_data.csv")
X = [{key: value for key, value in row.items() if key != "Usage_kWh"} for row in rows]
y = [row["Usage_kWh"] for row in rows]

split_index = int(len(rows) * 0.8)
X_train = X[:split_index]
X_test = X[split_index:]
y_train = y[:split_index]
y_test = y[split_index:]

vectorizer = DictVectorizer(sparse=False)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LinearRegression()
model.fit(X_train_vec, y_train)
predictions = model.predict(X_test_vec)
        


## Teilaufgabe 4.1
Eine lineare Regression hat keine eingebauten Baum-Wichtigkeiten wie ein Random Forest. Darum bestimme ich die informativen Felder mit **Permutationswichtigkeit**. Dabei wird jedes Merkmal einzeln durchmischt, und ich messe, wie stark sich der Fehler verschlechtert. Wenn der Fehler stark ansteigt, ist das Feld für die Vorhersage besonders wichtig.
        


In [2]:
importance_result = permutation_importance(
    model,
    X_test_vec,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error",
)

feature_names = vectorizer.get_feature_names_out()
ranking = sorted(
    zip(feature_names, importance_result.importances_mean),
    key=lambda item: item[1],
    reverse=True,
)

for name, score in ranking[:8]:
    print(f"{name:40s} {score:8.4f}")
        


Lagging_Current_Reactive.Power_kVarh      35.8740
Lagging_Current_Power_Factor              12.1859
Leading_Current_Power_Factor               7.9287
Hour                                       4.8947
NSM                                        1.7524
Leading_Current_Reactive_Power_kVarh       1.5259
Day                                        0.0638
Day_of_week=Friday                         0.0000


Am wichtigsten ist `Lagging_Current_Reactive.Power_kVarh`. Danach folgen vor allem `Lagging_Current_Power_Factor`, `Leading_Current_Power_Factor`, `Hour` und `NSM`. Das ist fachlich nachvollziehbar, weil Blindleistung, Leistungsfaktoren und Tageszeit eng mit dem Stromverbrauch einer Industrieanlage zusammenhängen. Der Kalendertag und einzelne Wochentage spielen dagegen in diesem einfachen Modell nur eine kleine Rolle.
        


## Teilaufgabe 4.2
Als Hauptmetrik wähle ich den **Mean Absolute Error (MAE)**. Er zeigt direkt, um wie viele kWh die Vorhersagen im Durchschnitt danebenliegen. Das ist leicht verständlich und passt gut zu einem Verbrauchsdatensatz. Zusätzlich berechne ich den R2-Wert, damit ich einschätzen kann, wie gut das Modell die allgemeinen Schwankungen des Verbrauchs erklärt.
        


In [3]:
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.4f} kWh")
print(f"R2: {r2:.4f}")
        


MAE: 9.7114 kWh
R2: 0.9412


## Teilaufgabe 4.3
Für die Wahrheitsmatrix formuliere ich eine klare Bedingung: Ein Verbrauch gilt als **positiv**, wenn er mindestens `90 kWh` beträgt. Damit unterscheide ich sehr hohe Lastphasen von normalen Messpunkten. Diese Schwelle ist sinnvoll, weil gerade hohe Lastspitzen für ein einfaches lineares Modell schwieriger vorherzusagen sind als durchschnittliche Verbräuche.
        


In [4]:
threshold = 90.0
y_true_binary = [1 if value >= threshold else 0 for value in y_test]
y_pred_binary = [1 if value >= threshold else 0 for value in predictions]

tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")
print(f"Sensitivität: {sensitivity:.4f}")
print(f"Spezifität: {specificity:.4f}")
        


TN = 117, FP = 2, FN = 21, TP = 60
Sensitivität: 0.7407
Spezifität: 0.9832


## Teilaufgabe 4.4
Das Modell funktioniert insgesamt ordentlich, aber nicht perfekt. Mit einem MAE von ungefähr `9.71 kWh` und einem R2 von rund `0.94` erkennt es die allgemeinen Muster des Verbrauchs recht gut. Die Spezifität ist hoch, also werden normale Phasen meist korrekt erkannt. Schwieriger sind sehr hohe Lastspitzen: Dort sinkt die Sensitivität, weil die lineare Regression extreme Werte eher glättet.
        


## Teilaufgabe 4.5
Die Adresse des Repository ist unverändert. Deshalb ist an dieser Stelle keine weitere Aktion nötig.
        
